# 🚢 Kaggle Titanic 竞赛入门 - Day 4

**作者**: 南风  
**日期**: 2026-03-26  
**学习目标**:
- 了解 Kaggle 竞赛流程
- 完成数据探索性分析 (EDA)
- 建立第一个基线模型
- 提交第一次预测结果

## 📦 1. 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体（如果需要）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 忽略警告
import warnings
warnings.filterwarnings('ignore')

print("所有库导入成功！✅")

## 📊 2. 加载数据

In [ ]:
# 从 Kaggle 下载数据后放在这里
# 或者直接从 CSV 文件加载
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"训练集形状：{train.shape}")
print(f"测试集形状：{test.shape}")

# 查看前几行
train.head()

## 🔍 3. 数据探索性分析 (EDA)

In [ ]:
# 查看数据基本信息
print("=" * 50)
print("训练集信息摘要")
print("=" * 50)
train.info()

print("\n" + "=" * 50)
print("缺失值统计")
print("=" * 50)
print(train.isnull().sum())

In [ ]:
# 描述性统计
train.describe()

In [ ]:
# 生存率统计
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 生存/死亡人数
axes[0].bar(['死亡', '生存'], train['Survived'].value_counts().sort_index())
axes[0].set_title('生存 vs 死亡')
axes[0].set_ylabel('人数')

# 生存率
survival_rate = train['Survived'].mean() * 100
axes[1].pie([survival_rate, 100-survival_rate], 
            labels=['生存', '死亡'], 
            autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'])
axes[1].set_title(f'生存率：{survival_rate:.1f}%')

plt.tight_layout()
plt.show()

print(f"总生存率：{survival_rate:.2f}%")

In [ ]:
# 按性别分析生存率
sex_survival = train.groupby('Sex')['Survived'].mean()
print(sex_survival)

plt.figure(figsize=(8, 5))
sex_survival.plot(kind='bar', color=['#3498db', '#e91e63'])
plt.title('不同性别的生存率')
plt.ylabel('生存率')
plt.xticks(rotation=0)
for i, v in enumerate(sex_survival):
    plt.text(i, v + 0.02, f'{v:.2%}', ha='center')
plt.ylim(0, 1)
plt.show()

print(f"\n女性生存率：{sex_survival['female']:.2%}")
print(f"男性生存率：{sex_survival['male']:.2%}")

In [ ]:
# 按船票等级分析生存率
pclass_survival = train.groupby('Pclass')['Survived'].mean()
print(pclass_survival)

plt.figure(figsize=(8, 5))
pclass_survival.plot(kind='bar', color=['#f39c12', '#9b59b6', '#1abc9c'])
plt.title('不同船票等级的生存率')
plt.ylabel('生存率')
plt.xticks(rotation=0)
for i, v in enumerate(pclass_survival):
    plt.text(i, v + 0.02, f'{v:.2%}', ha='center')
plt.ylim(0, 1)
plt.show()

print(f"\n头等舱 (1 等) 生存率：{pclass_survival[1]:.2%}")
print(f"二等舱生存率：{pclass_survival[2]:.2%}")
print(f"三等舱 (3 等) 生存率：{pclass_survival[3]:.2%}")

## 🔧 4. 特征工程

In [ ]:
def preprocess_data(df):
    """数据预处理函数"""
    df = df.copy()
    
    # 1. 填充缺失值
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    df['Embarked'].fillna('S', inplace=True)
    
    # 2. 性别编码
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    
    # 3. 登船港口编码
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    
    # 4. 家庭大小特征
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    return df

# 应用预处理
train_processed = preprocess_data(train)
test_processed = preprocess_data(test)

print("数据预处理完成！✅")
print(f"处理后的训练集形状：{train_processed.shape}")

## 🤖 5. 建立基线模型

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 选择特征
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']
X = train_processed[features]
y = train_processed['Survived']

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"训练集大小：{len(X_train)}")
print(f"验证集大小：{len(X_val)}")

In [ ]:
# 训练随机森林模型
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# 在验证集上评估
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"验证集准确率：{accuracy:.4f} ({accuracy:.2%})")
print("\n分类报告:")
print(classification_report(y_val, y_pred, target_names=['死亡', '生存']))

In [ ]:
# 特征重要性
feature_importance = pd.DataFrame({
    '特征': features,
    '重要性': model.feature_importances_
}).sort_values('重要性', ascending=False)

print(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['特征'], feature_importance['重要性'])
plt.xlabel('重要性')
plt.title('特征重要性排名')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 📤 6. 生成提交文件

In [ ]:
# 用全部训练数据重新训练模型
model.fit(X, y)

# 预测测试集
X_test = test_processed[features]
predictions = model.predict(X_test)

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

# 保存
submission.to_csv('titanic_submission.csv', index=False)
print("提交文件已生成：titanic_submission.csv ✅")
print(f"\n前 10 行预览:")
submission.head(10)

## 🎯 7. 下一步优化方向

### 可以尝试的改进：

1. **特征工程**
   - 从姓名中提取称谓 (Title)
   - 年龄分箱 (儿童/成人/老人)
   - 票价分箱

2. **模型调参**
   - 使用 GridSearchCV 优化超参数
   - 尝试不同的模型 (Logistic Regression, XGBoost, etc.)

3. **集成学习**
   - 多模型投票
   - Stacking

4. **处理异常值**
   - 检查 Fare 的分布
   - 处理极端值

## 📝 总结

- ✅ 完成了数据探索性分析
- ✅ 发现女性和头等舱乘客生存率更高
- ✅ 建立了随机森林基线模型
- ✅ 生成了提交文件

**下一步**: 
1. 登录 Kaggle 提交预测结果
2. 查看得分和排名
3. 根据反馈继续优化模型